In [ ]:
CREATE CATALOG IF NOT EXISTS investment_platform;

USE CATALOG investment_platform;

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

SHOW SCHEMAS;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.bronze.accounts_raw
USING DELTA
AS
SELECT
    CONCAT('ACC', LPAD(CAST(id AS STRING), 6, '0')) AS account_id,

    CASE
        WHEN id % 250 = 0 THEN NULL
        ELSE CONCAT('Customer_', id)
    END AS customer_name,

    CASE
        WHEN id % 3 = 0 THEN 'RETIREMENT'
        WHEN id % 3 = 1 THEN 'INDIVIDUAL'
        ELSE 'JOINT'
    END AS account_type,

    CASE
        WHEN id % 10 = 0 THEN 'INACTIVE'
        ELSE 'ACTIVE'
    END AS account_status,

    DATE_SUB(CURRENT_DATE(), CAST(id % 1825 AS INT)) AS opened_date,

    CURRENT_TIMESTAMP() AS ingestion_timestamp

FROM RANGE(1, 1001);

In [ ]:
CREATE OR REPLACE TABLE investment_platform.bronze.securities_raw
USING DELTA
AS
SELECT
    security_id,
    symbol,
    security_name,
    asset_type,
    sector,
    currency,
    CURRENT_TIMESTAMP() AS ingestion_timestamp
FROM VALUES
    ('SEC001', 'AAPL',  'Apple Inc.',             'STOCK', 'Technology',        'USD'),
    ('SEC002', 'MSFT',  'Microsoft Corporation',  'STOCK', 'Technology',        'USD'),
    ('SEC003', 'GOOGL', 'Alphabet Inc.',           'STOCK', 'Technology',        'USD'),
    ('SEC004', 'AMZN',  'Amazon.com Inc.',         'STOCK', 'Consumer',          'USD'),
    ('SEC005', 'TSLA',  'Tesla Inc.',              'STOCK', 'Automotive',        'USD'),
    ('SEC006', 'JPM',   'JPMorgan Chase & Co.',    'STOCK', 'Financial Services','USD'),
    ('SEC007', 'V',     'Visa Inc.',               'STOCK', 'Financial Services','USD'),
    ('SEC008', 'JNJ',   'Johnson & Johnson',       'STOCK', 'Healthcare',        'USD'),
    ('SEC009', 'XOM',   'Exxon Mobil Corporation', 'STOCK', 'Energy',            'USD'),
    ('SEC010', 'NVDA',  'NVIDIA Corporation',      'STOCK', 'Technology',        'USD'),
    ('SEC011', 'SPY',   'S&P 500 ETF',             'ETF',   'Diversified',       'USD'),
    ('SEC012', 'BND',   'Total Bond Market ETF',   'ETF',   'Fixed Income',      'USD')
AS securities(
    security_id,
    symbol,
    security_name,
    asset_type,
    sector,
    currency
);

In [ ]:
SELECT *
FROM investment_platform.bronze.securities_raw;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.bronze.trades_raw
USING DELTA
AS
SELECT
    CONCAT(
        'TRD',
        LPAD(CAST(id AS STRING), 8, '0')
    ) AS trade_id,

    CONCAT(
        'ACC',
        LPAD(CAST((id % 1000) + 1 AS STRING), 6, '0')
    ) AS account_id,

    CONCAT(
        'SEC',
        LPAD(CAST((id % 12) + 1 AS STRING), 3, '0')
    ) AS security_id,

    CASE
        WHEN id % 10 < 7 THEN 'BUY'
        ELSE 'SELL'
    END AS trade_type,

    CAST((id % 100) + 1 AS INT) AS quantity,

    ROUND(50 + (id % 450) + RAND(42), 2) AS trade_price,

    TIMESTAMPADD(
        MINUTE,
        -CAST(id % 43200 AS INT),
        CURRENT_TIMESTAMP()
    ) AS trade_timestamp,

    CURRENT_TIMESTAMP() AS ingestion_timestamp

FROM RANGE(1, 50001);

In [ ]:
SELECT COUNT(*) AS total_trades
FROM investment_platform.bronze.trades_raw;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.bronze.market_prices_raw
USING DELTA
AS
SELECT
    CONCAT(
        'SEC',
        LPAD(CAST(s.id AS STRING), 3, '0')
    ) AS security_id,

    ROUND(
        50
        + (s.id * 20)
        + ((h.id % 24) * 0.25)
        + RAND(99),
        2
    ) AS market_price,

    TIMESTAMPADD(
        HOUR,
        -CAST(h.id AS INT),
        CURRENT_TIMESTAMP()
    ) AS price_timestamp,

    CAST(
        10000 + ((s.id * (h.id + 1) * 137) % 900000)
        AS BIGINT
    ) AS trading_volume,

    CURRENT_TIMESTAMP() AS ingestion_timestamp

FROM RANGE(1, 13) AS s

CROSS JOIN RANGE(0, 720) AS h;

In [ ]:
SELECT * FROM  investment_platform.bronze.market_prices_raw;


In [ ]:
CREATE OR REPLACE TABLE investment_platform.silver.accounts_clean
USING DELTA
AS
SELECT
    account_id,
    customer_name,
    account_type,
    account_status,
    opened_date,
    ingestion_timestamp
FROM investment_platform.bronze.accounts_raw
WHERE customer_name IS NOT NULL
  AND account_type IN ('INDIVIDUAL', 'JOINT', 'RETIREMENT')
  AND account_status IN ('ACTIVE', 'INACTIVE');

In [ ]:
SELECT * FROM investment_platform.silver.accounts_clean

In [ ]:
CREATE OR REPLACE TABLE investment_platform.silver.accounts_rejected
USING DELTA
AS
SELECT
    *,
    CASE
        WHEN customer_name IS NULL THEN 'MISSING CUSTOMER NAME'
        WHEN account_type IS NULL THEN 'MISSING ACCOUNT TYPE'
        WHEN account_type NOT IN ('INDIVIDUAL', 'JOINT', 'RETIREMENT')
            THEN 'INVALID ACCOUNT TYPE'
        WHEN account_status IS NULL THEN 'MISSING ACCOUNT STATUS'
        WHEN account_status NOT IN ('ACTIVE', 'INACTIVE')
            THEN 'INVALID ACCOUNT STATUS'
    END AS rejection_reason
FROM investment_platform.bronze.accounts_raw
WHERE customer_name IS NULL
   OR account_type IS NULL
   OR account_type NOT IN ('INDIVIDUAL', 'JOINT', 'RETIREMENT')
   OR account_status IS NULL
   OR account_status NOT IN ('ACTIVE', 'INACTIVE');

In [ ]:
CREATE OR REPLACE TABLE investment_platform.silver.securities_clean
USING DELTA
AS
SELECT DISTINCT
    security_id,
    symbol,
    security_name,
    asset_type,
    sector,
    currency,
    ingestion_timestamp
FROM investment_platform.bronze.securities_raw
WHERE security_id IS NOT NULL
  AND symbol IS NOT NULL
  AND security_name IS NOT NULL
  AND asset_type IN ('STOCK', 'ETF')
  AND currency = 'USD';

In [ ]:
SELECT *
FROM investment_platform.silver.securities_clean;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.silver.trades_clean
USING DELTA
AS
SELECT DISTINCT
    t.trade_id,
    t.account_id,
    t.security_id,
    t.trade_type,
    t.quantity,
    t.trade_price,
    t.trade_timestamp,
    t.ingestion_timestamp
FROM investment_platform.bronze.trades_raw AS t

INNER JOIN investment_platform.silver.accounts_clean AS a
    ON t.account_id = a.account_id

INNER JOIN investment_platform.silver.securities_clean AS s
    ON t.security_id = s.security_id

WHERE t.trade_id IS NOT NULL
  AND t.trade_type IN ('BUY', 'SELL')
  AND t.quantity > 0
  AND t.trade_price > 0;

In [ ]:
SELECT * FROM  investment_platform.silver.trades_clean;


In [ ]:
CREATE OR REPLACE TABLE investment_platform.silver.trades_rejected
USING DELTA
AS
SELECT
    t.*,
    CASE
        WHEN t.trade_id IS NULL
            THEN 'MISSING TRADE ID'
        WHEN a.account_id IS NULL
            THEN 'INVALID OR REJECTED ACCOUNT'
        WHEN s.security_id IS NULL
            THEN 'INVALID SECURITY'
        WHEN t.trade_type NOT IN ('BUY', 'SELL')
            THEN 'INVALID TRADE TYPE'
        WHEN t.quantity <= 0
            THEN 'INVALID QUANTITY'
        WHEN t.trade_price <= 0
            THEN 'INVALID TRADE PRICE'
    END AS rejection_reason
FROM investment_platform.bronze.trades_raw AS t
LEFT JOIN investment_platform.silver.accounts_clean AS a
    ON t.account_id = a.account_id
LEFT JOIN investment_platform.silver.securities_clean AS s
    ON t.security_id = s.security_id
WHERE t.trade_id IS NULL
   OR a.account_id IS NULL
   OR s.security_id IS NULL
   OR t.trade_type NOT IN ('BUY', 'SELL')
   OR t.quantity <= 0
   OR t.trade_price <= 0;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.silver.market_prices_clean
USING DELTA
AS
SELECT DISTINCT
    m.security_id,
    m.market_price,
    m.price_timestamp,
    m.trading_volume,
    m.ingestion_timestamp
FROM investment_platform.bronze.market_prices_raw AS m

INNER JOIN investment_platform.silver.securities_clean AS s
    ON m.security_id = s.security_id

WHERE m.market_price > 0
  AND m.trading_volume >= 0
  AND m.price_timestamp IS NOT NULL;

In [ ]:
SELECT
    COUNT(*) AS total_records,

    SUM(
        CASE WHEN market_price <= 0
        THEN 1 ELSE 0 END
    ) AS invalid_prices,

    SUM(
        CASE WHEN trading_volume < 0
        THEN 1 ELSE 0 END
    ) AS invalid_volumes,

    SUM(
        CASE WHEN price_timestamp IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_timestamps

FROM investment_platform.silver.market_prices_clean;

In [ ]:
DESCRIBE TABLE investment_platform.silver.trades_clean;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.gold.daily_trade_summary
USING DELTA
AS

SELECT
    DATE(trade_timestamp) AS trade_date,
    security_id,

    COUNT(*) AS total_trades,

    SUM(
        CASE
            WHEN trade_type = 'BUY' THEN quantity
            ELSE 0
        END
    ) AS total_buy_quantity,

    SUM(
        CASE
            WHEN trade_type = 'SELL' THEN quantity
            ELSE 0
        END
    ) AS total_sell_quantity,

    ROUND(
        SUM(quantity * trade_price),
        2
    ) AS total_trade_value

FROM investment_platform.silver.trades_clean

GROUP BY
    DATE(trade_timestamp),
    security_id;

In [ ]:
SELECT *
FROM investment_platform.gold.daily_trade_summary
ORDER BY
    trade_date,
    security_id
LIMIT 20;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.gold.latest_market_prices
USING DELTA
AS

WITH ranked_prices AS
(
    SELECT
        security_id,
        market_price,
        price_timestamp,
        trading_volume,

        ROW_NUMBER() OVER (
            PARTITION BY security_id
            ORDER BY price_timestamp DESC
        ) AS price_rank

    FROM investment_platform.silver.market_prices_clean
)

SELECT
    security_id,
    market_price AS latest_market_price,
    price_timestamp AS latest_price_timestamp,
    trading_volume

FROM ranked_prices

WHERE price_rank = 1;


In [ ]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT security_id) AS unique_securities

FROM investment_platform.gold.latest_market_prices;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.gold.portfolio_positions
USING DELTA
AS

WITH position_quantities AS
(
    SELECT
        account_id,
        security_id,

        SUM(
            CASE
                WHEN trade_type = 'BUY' THEN quantity
                WHEN trade_type = 'SELL' THEN -quantity
                ELSE 0
            END
        ) AS net_quantity

    FROM investment_platform.silver.trades_clean

    GROUP BY
        account_id,
        security_id
)

SELECT
    p.account_id,
    p.security_id,
    p.net_quantity,
    m.latest_market_price,

    ROUND(
        p.net_quantity * m.latest_market_price,
        2
    ) AS current_market_value,

    m.latest_price_timestamp

FROM position_quantities AS p

INNER JOIN investment_platform.gold.latest_market_prices AS m
    ON p.security_id = m.security_id

WHERE p.net_quantity <> 0;

In [ ]:
SELECT *
FROM investment_platform.gold.portfolio_positions
ORDER BY current_market_value DESC
LIMIT 20;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.gold.account_portfolio_summary
USING DELTA
AS

SELECT
    account_id,

    COUNT(*) AS total_positions,

    ROUND(
        SUM(current_market_value),
        2
    ) AS total_portfolio_value,

    ROUND(
        AVG(current_market_value),
        2
    ) AS average_position_value,

    ROUND(
        MAX(current_market_value),
        2
    ) AS largest_position_value

FROM investment_platform.gold.portfolio_positions

GROUP BY account_id;

In [ ]:
SELECT *
FROM investment_platform.gold.account_portfolio_summary
ORDER BY total_portfolio_value DESC
LIMIT 10;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.gold.security_exposure_summary
USING DELTA
AS

SELECT
    security_id,

    COUNT(DISTINCT account_id) AS total_accounts,

    SUM(net_quantity) AS total_net_quantity,

    ROUND(
        SUM(current_market_value),
        2
    ) AS total_market_value,

    ROUND(
        AVG(current_market_value),
        2
    ) AS average_position_value,

    ROUND(
        MAX(current_market_value),
        2
    ) AS largest_account_position

FROM investment_platform.gold.portfolio_positions

GROUP BY security_id;

In [ ]:
SELECT *
FROM investment_platform.gold.security_exposure_summary
ORDER BY ABS(total_market_value) DESC
LIMIT 10;

In [ ]:
DESCRIBE TABLE investment_platform.silver.securities_clean;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.gold.security_exposure_report
USING DELTA
AS

SELECT
    e.security_id,
    s.symbol,
    s.security_name,
    s.asset_type,
    s.sector,
    s.currency,
    e.total_accounts,
    e.total_net_quantity,
    e.total_market_value,
    e.average_position_value,
    e.largest_account_position

FROM investment_platform.gold.security_exposure_summary AS e

INNER JOIN investment_platform.silver.securities_clean AS s
    ON e.security_id = s.security_id;

In [ ]:
CREATE OR REPLACE TABLE investment_platform.gold.sector_exposure_summary
USING DELTA
AS

SELECT
    s.sector,

    COUNT(DISTINCT p.security_id) AS total_securities,

    COUNT(DISTINCT p.account_id) AS total_accounts,

    ROUND(
        SUM(p.current_market_value),
        2
    ) AS total_sector_value,

    ROUND(
        AVG(p.current_market_value),
        2
    ) AS average_position_value,

    ROUND(
        MAX(p.current_market_value),
        2
    ) AS largest_position_value

FROM investment_platform.gold.portfolio_positions AS p

INNER JOIN investment_platform.silver.securities_clean AS s
    ON p.security_id = s.security_id

GROUP BY s.sector;

In [ ]:
SELECT * FROM investment_platform.gold.sector_exposure_summary;


In [ ]:
SELECT *
FROM investment_platform.gold.sector_exposure_summary
ORDER BY ABS(total_sector_value) DESC;